# CER — Cluster Experiment Runner Demo

This notebook demonstrates the full CER workflow via the **MCP server**.

Architecture:
- `cer-mcp` runs on the host with SSH access to the cluster
- The agent (or this notebook) connects via MCP over SSE and calls tools
- Available tools: `submit`, `status`, `cancel`, `results`, `list_experiments`

## 0. Start the MCP Server

In a separate terminal, run:
```bash
cer-mcp
```

This starts the server on `http://localhost:8000/sse`.

In [ ]:
from mcp.client.session import ClientSession
from mcp.client.sse import sse_client
import json

MCP_URL = "http://localhost:8000/sse"

In [ ]:
# Helper to run MCP calls from sync notebook cells
import asyncio
import nest_asyncio
nest_asyncio.apply()

async def call_tool(name: str, args: dict = {}) -> str:
    """Connect to the CER MCP server and call a tool."""
    async with sse_client(MCP_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(name, args)
            return result.content[0].text

async def get_tools() -> list:
    """List all available tools on the CER MCP server."""
    async with sse_client(MCP_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()
            return result.tools

## 1. Discover Available Tools

In [ ]:
tools = asyncio.get_event_loop().run_until_complete(get_tools())
for t in tools:
    print(f"  {t.name:20s} — {t.description.splitlines()[0]}")

## 2. Commit & Push

CER uses **commit = experiment**. Every experiment is tied to a git commit for full reproducibility.

In [ ]:
import subprocess

!git status
print("---")
!git log --oneline -5

In [ ]:
# Uncomment to commit and push:
# !git add -A && git commit -m "experiment: describe changes" && git push

In [ ]:
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"Commit: {COMMIT}")

## 3. Submit Experiment

Calls `submit` tool on the MCP server → SSH to cluster → create worktree → sbatch.

In [ ]:
result = asyncio.get_event_loop().run_until_complete(
    call_tool("submit", {"commit_hash": COMMIT})
)
print(result)

# Extract job ID
JOB_ID = result.split("Job ID: ")[-1].strip()
print(f"\nJob ID: {JOB_ID}")

## 4. Check Status

Queries SLURM via SSH. Run multiple times to watch: `PENDING` → `RUNNING` → `COMPLETED`.

In [ ]:
result = asyncio.get_event_loop().run_until_complete(
    call_tool("status", {"job_id": JOB_ID})
)
status_data = json.loads(result)
print(json.dumps(status_data, indent=2))

## 5. List All Experiments

Returns all tracked experiments. Active jobs get their SLURM status refreshed.

In [ ]:
result = asyncio.get_event_loop().run_until_complete(
    call_tool("list_experiments")
)
experiments = json.loads(result)
for exp in experiments:
    print(f"  {exp['job_id']}  {exp['commit_short']}  {exp['status']:10s}  {exp['submitted_at'][:19]}")

## 6. Query W&B Results

Finds the W&B run by commit hash tag, pulls summary and history.

In [ ]:
# Summary only
result = asyncio.get_event_loop().run_until_complete(
    call_tool("results", {"job_id": JOB_ID})
)
data = json.loads(result)
print(f"Run:   {data['wandb']['run_name']} ({data['wandb']['state']})")
print(f"URL:   {data['wandb']['url']}")
print(f"\nSummary:")
for k, v in data['wandb']['summary'].items():
    print(f"  {k}: {v}")

In [ ]:
# Full history
result = asyncio.get_event_loop().run_until_complete(
    call_tool("results", {"job_id": JOB_ID, "history": True})
)
data = json.loads(result)
print(f"History: {len(data.get('history', []))} steps")

In [ ]:
# Filtered metrics
result = asyncio.get_event_loop().run_until_complete(
    call_tool("results", {"job_id": JOB_ID, "history": True, "keys": ["train/loss", "val/loss"]})
)
data = json.loads(result)
for step in data["history"][:5]:
    print(step)

## 7. Plot Results

In [ ]:
import matplotlib.pyplot as plt

history = data["history"]
steps = range(len(history))
train_loss = [h.get("train/loss") for h in history]
val_loss = [h.get("val/loss") for h in history]

plt.figure(figsize=(10, 5))
if any(v is not None for v in train_loss):
    plt.plot(steps, train_loss, label="train/loss", marker="o")
if any(v is not None for v in val_loss):
    plt.plot(steps, val_loss, label="val/loss", marker="s")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title(f"Experiment {JOB_ID} (commit {COMMIT[:8]})")
plt.legend()
plt.grid(True)
plt.show()

## 8. Cancel an Experiment

In [ ]:
# Uncomment to cancel:
# result = asyncio.get_event_loop().run_until_complete(
#     call_tool("cancel", {"job_id": JOB_ID})
# )
# print(result)

---

## Tool Reference

All interactions go through the MCP server (`cer-mcp`).

| Tool | Args | Description |
|------|------|-------------|
| `submit` | `commit_hash` | Submit experiment to SLURM cluster |
| `status` | `job_id` | Check SLURM job status (returns JSON) |
| `cancel` | `job_id` | Cancel a running job |
| `results` | `job_id`, `history?`, `keys?` | Query W&B metrics |
| `list_experiments` | — | List all tracked experiments |